<a href="https://colab.research.google.com/github/Rusira54321/SimpleRagApplication/blob/main/simpleRagApplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install langchain -qU
!pip install langchain-openai -qU
!pip install langchain-chroma -qU

In [25]:
# import necessary libraries
import os
from google.colab import userdata

In [26]:
from langchain_openai import ChatOpenAI
os.environ['OPENAI_API_KEY'] = userdata.get('OpenAi_Key')
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [27]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [28]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="The T20 World Cup 2024 is in full swing, bringing excitement and drama to cricket fans worldwide.India's team, captained by Rohit Sharma, is preparing for a crucial match against Ireland, with standout player Jasprit Bumrah expected to play a pivotal role in their campaign.The tournament has already seen controversy, particularly concerning the pitch conditions at Nassau County International Cricket Stadium in New York, which came under fire after a low-scoring game between Sri Lanka and South Africa.",
        metadata={"source": "cricket news"},
    ),
    Document(
        page_content="The world of football is buzzing with excitement as major tournaments and league matches continue to captivate fans globally.In the UEFA Champions League, the semi-final matchups have been set, with defending champions Real Madrid set to face Manchester City, while Bayern Munich will take on Paris Saint-Germain.Both ties promise thrilling encounters, featuring some of the best talents in world football.",
        metadata={"source": "football news"},
    ),
    Document(
        page_content="As election season heats up, the latest developments reveal a highly competitive atmosphere across several key races.The presidential election has seen intense campaigning from all major candidates, with recent polls indicating a tight race.Incumbent President Jane Doe is seeking re-election on a platform of economic stability and healthcare reform, while her main rival, Senator John Smith, focuses on education and climate change initiatives.",
        metadata={"source": "election news"},
    ),
    Document(
        page_content="The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.",
        metadata={"source": "ai revolution news"},
    )
]

In [29]:
#create a vector store using the documents and embedding model
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents,
    embedding=embedding_model
)

In [30]:
results = vectorstore.similarity_search("nlp")
for result in results:
  print("_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _")
  print(result.page_content)
  print(result.metadata)

_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.
{'source': 'ai revolution news'}
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
As election season heats up, the latest developments reveal a highly competitive atmosphere across

In [31]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1},
)

In [32]:
result = retriever.batch(["machine learning"])
print(result)

[[Document(id='7567ffbf-5cb1-4be1-93a9-263dddddaf18', metadata={'source': 'ai revolution news'}, page_content='The AI revolution continues to transform industries and reshape the global economy.Significant advancements in artificial intelligence have led to breakthroughs in healthcare, with AI-driven diagnostics improving patient outcomes and reducing costs.Autonomous systems are becoming increasingly prevalent in logistics and transportation, enhancing efficiency and safety.')]]


In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

In [34]:
chain = {"context":retriever,"question":RunnablePassthrough()} | prompt | llm

In [37]:
response = chain.invoke("who is the captain of indias cricket team")
print(response.content)

The captain of India's cricket team is Rohit Sharma.
